# PHASE-HOPFIELD-IR — Non-Commutative Fractional Hopfield Dynamics for All-in-One Image Restoration
### Thực Nghiệm Khóa Trước Đối Chứng Độc Lập (Pilot v3 — Tesla T4 GPU)

**Trụ Cột Toán Học & Đột Phá Khoa Học**:
1. **Giải tích Phân số (Fractional Fourier Transform - FrFT)**: Không gian pha liên tục $(x, p)_\alpha$ với góc quay thích ứng $\alpha(x) \in (0, 1)$.
2. **Đại số Lie (Lie Algebra Commutator)**: Đo đạc chính xác số hạng tương tác chéo phi giao hoán $[\mathcal{O}_{\text{haze}}, \mathcal{O}_{\text{noise}}] = \epsilon(1 - t)$ bị các mô hình phẳng bỏ sót.
3. **Vật lý Thống kê Hopfield (Giải Nobel Vật lý 2024)**: Dòng chảy gradient theo thế năng Lyapunov $E(\mathbf{z})$ kéo quỹ đạo về manifold ảnh sạch.
4. **Triệt tiêu lỗi ngủ đông (Dormant Block)**: Ghép năng lượng Energy-Scaled Coupling + Tách nhóm tối ưu Dual Learning Rate ($lr_{\text{novel}} = 10 \times lr_{\text{base}}$).

**Tiêu Chuẩn Quyết Định Khoa Học (Preregistered Gate P5)**:
- $\Delta(B - A) = \mu_B - \mu_A \ge X = \max(0.40, 2\hat{\sigma}_A)$ dB trên Held-out Hybrid-OOD.
- $\Delta(B - C) = \mu_B - \mu_C \ge Z = \max(0.25, 1.5\hat{\sigma}_A)$ dB (vượt trội mô hình đối chứng tham số).
- Tỉ lệ năng lượng khối mới `Ratio(New/Base)` $\ge 15\%$.
- Tác vụ đơn lẻ (Noise, Rain, Haze) không suy giảm quá $0.15$ dB.

## 1. Thiết Lập Môi Trường & Kiểm Tra GPU Tesla T4

In [ ]:
# Kiểm tra phần cứng GPU và phiên bản CUDA
!nvidia-smi

import torch
print(f"PyTorch: {torch.__version__} | CUDA Sẵn sàng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB VRAM)")


## 2. Đồng Bộ Codebase Sang Đĩa SSD Cục Bộ Colab

In [ ]:
import os, shutil
from pathlib import Path

%cd /content
!rm -rf /content/repo /content/temp_repo

print("Đang clone codebase từ GitHub...")
!git clone https://github.com/nthmaiii/all-in-one-image-restoration.git /content/temp_repo

# Chuyển thư mục repo ra /content/repo và giữ .git để phục vụ audit commit HEAD
shutil.move("/content/temp_repo/repo", "/content/repo")
shutil.move("/content/temp_repo/.git", "/content/repo/.git")
!rm -rf /content/temp_repo

%cd /content/repo
!pip install -q -r requirements.txt

# Kiểm toán Git Commit HEAD
commit_hash = !git rev-parse HEAD
print(f">> [AUDIT KHOA HỌC] Git Commit HEAD: {commit_hash[0]}")
os.makedirs("runs/phase_hopfield_v3", exist_ok=True)
with open("runs/phase_hopfield_v3/git_commit.txt", "w", encoding="utf-8") as f:
    f.write(commit_hash[0] + "\n")
print(">> Môi trường thực thi PHASE-HOPFIELD-IR đã sẵn sàng!")


## 3. Tải Dữ Liệu Kiểm Chuẩn & Khẳng Định Chống Rò Rỉ (Zero-Leakage Audit)

In [ ]:
from pathlib import Path
if Path("/content/repo/repo/scripts").exists():
    !cp -rf /content/repo/repo/* /content/repo/
    !rm -rf /content/repo/repo

# 1. Tải checkpoint PromptIR All-in-One và tập test held-out (BSD68, Kodak24)
!python scripts/download_datasets.py --pilot_v2

# 2. Chuẩn bị 160 ảnh train và 40 ảnh val sạch độc lập & kiểm tra zero-leakage fail-closed
!python scripts/prepare_data_v2.py


## 4. Kiểm Chứng Toán Học & Toàn Bộ Bộ Unit Test (Gate 1 & Gate 3 Pass)

In [ ]:
# 1. Đo đạc thực nghiệm Giao hoán tử Lie [Koschmieder, Gaussian] = eps * (1 - t) trên ảnh thật
!python scripts/verify_lie_commutator_gap.py

# 2. Chạy 5/5 Unit Test kiểm toán toán học (Phản đối xứng Lie, Đơn điệu năng lượng Hopfield, Đẳng tham số B vs C)
!python tests/test_phase_hopfield.py


## 5. Smoke Test Trên GPU Tesla T4 (Kiểm Tra Toàn Bộ Pipeline Trước Huấn Luyện)

In [ ]:
# Smoke test nhánh A, B, C (iters=2) xác nhận DataLoader, Dual LR và Evaluator chạy mượt mà
!python scripts/train_phase_hopfield.py --branch A --smoke_test --seed 42 --patch_size 128
!python scripts/train_phase_hopfield.py --branch B --smoke_test --seed 42 --patch_size 128
!python scripts/train_phase_hopfield.py --branch C --smoke_test --seed 42 --patch_size 128
print(">> Smoke test hoàn tất thành công! Mọi pipeline đã xác nhận hoạt động 100% trên GPU.")


## 6. Nhánh A: PromptIR Baseline (3 Seeds: 42, 123, 2024)

In [ ]:
# Huấn luyện Nhánh A để thiết lập độ lệch chuẩn cơ sở và khóa ngưỡng thống kê
!python scripts/train_phase_hopfield.py --branch A --seed 42 --iters 1500 --batch_size 4 --patch_size 128
!python scripts/train_phase_hopfield.py --branch A --seed 123 --iters 1500 --batch_size 4 --patch_size 128
!python scripts/train_phase_hopfield.py --branch A --seed 2024 --iters 1500 --batch_size 4 --patch_size 128


## 7. Khóa Ngưỡng Thống Kê Thuyết Minh Dựa Trên σ_A (Preregistered Decision Gate P5)

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

csv_path = Path("results/phase_hopfield_results.csv")
assert csv_path.exists(), "Chưa tìm thấy file results/phase_hopfield_results.csv!"

df = pd.read_csv(csv_path)
df_a = df[(df["branch"] == "A") & (df["checkpoint_type"] == "best_val")]
assert len(df_a) == 3, f"Nhánh A phải đủ 3 seeds trước khi khóa ngưỡng! Hiện có: {len(df_a)}"

psnr_a = df_a["primary_endpoint_ood_psnr"].values
mu_a = float(np.mean(psnr_a))
sigma_a = float(np.std(psnr_a, ddof=1))

X = float(max(0.40, 2.0 * sigma_a))
Z = float(max(0.25, 1.5 * sigma_a))

thresholds = {
    "mu_A": round(mu_a, 4),
    "sigma_A": round(sigma_a, 4),
    "X_threshold_B_vs_A": round(X, 4),
    "Z_threshold_B_vs_C": round(Z, 4),
    "seeds_A": df_a["seed"].tolist(),
    "psnr_A_per_seed": [round(x, 4) for x in psnr_a.tolist()]
}

with open("thresholds_locked_v3.json", "w", encoding="utf-8") as f:
    json.dump(thresholds, f, indent=2)

print("=" * 65)
print("ĐÃ KHÓA NGƯỠNG THỐNG KÊ (PRE-REGISTERED THRESHOLDS LOCKED):")
print(f"-> Baseline PromptIR Mean (mu_A):  {mu_a:.4f} dB")
print(f"-> Baseline PromptIR Std (sigma_A): {sigma_a:.4f} dB")
print(f"-> Ngưỡng Vượt Trội B vs A (X):      {X:.4f} dB")
print(f"-> Ngưỡng Vượt Trội B vs C (Z):      {Z:.4f} dB")
print("=" * 65)


## 8. Nhánh B: Huấn Luyện Đề Xuất PHASE-HOPFIELD-IR (3 Seeds: 42, 123, 2024)

In [ ]:
# Huấn luyện mô hình đề xuất PhaseHopfieldIR với Dual Learning Rate
!python scripts/train_phase_hopfield.py --branch B --seed 42 --iters 1500 --batch_size 4 --patch_size 128 --lr 1e-4 --lr_novel 2e-4 --lambda_hop 0.0
!python scripts/train_phase_hopfield.py --branch B --seed 123 --iters 1500 --batch_size 4 --patch_size 128 --lr 1e-4 --lr_novel 2e-4 --lambda_hop 0.0
!python scripts/train_phase_hopfield.py --branch B --seed 2024 --iters 1500 --batch_size 4 --patch_size 128 --lr 1e-4 --lr_novel 2e-4 --lambda_hop 0.0


## 9. Nhánh C: Đối Chứng Tham Số Param-Matched Control (3 Seeds: 42, 123, 2024)

In [ ]:
# Huấn luyện mô hình đối chứng tham số (cùng số tham số, cùng Dual LR, không FrFT/Lie/Hopfield)
!python scripts/train_phase_hopfield.py --branch C --seed 42 --iters 1500 --batch_size 4 --patch_size 128 --lr 1e-4 --lr_novel 2e-4 --lambda_hop 0.0
!python scripts/train_phase_hopfield.py --branch C --seed 123 --iters 1500 --batch_size 4 --patch_size 128 --lr 1e-4 --lr_novel 2e-4 --lambda_hop 0.0
!python scripts/train_phase_hopfield.py --branch C --seed 2024 --iters 1500 --batch_size 4 --patch_size 128 --lr 1e-4 --lr_novel 2e-4 --lambda_hop 0.0


## 10. Tổng Hợp Kết Quả & Nghiệm Thu Khoa Học Quyết Định (Decision Gate P5)

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from google.colab import files

csv_path = Path("results/phase_hopfield_results.csv")
assert csv_path.exists(), "Không tìm thấy results/phase_hopfield_results.csv!"
df = pd.read_csv(csv_path)

with open("thresholds_locked_v3.json", "r", encoding="utf-8") as f:
    thresh = json.load(f)
X = thresh["X_threshold_B_vs_A"]
Z = thresh["Z_threshold_B_vs_C"]

df_best = df[df["checkpoint_type"] == "best_val"]

res = {}
for br in ["A", "B", "C"]:
    sub = df_best[df_best["branch"] == br]
    psnr_vals = sub["primary_endpoint_ood_psnr"].values
    ratio_vals = sub["new_block_ratio"].values if "new_block_ratio" in sub else [0.0]*len(sub)
    res[br] = {
        "mean": float(np.mean(psnr_vals)),
        "std": float(np.std(psnr_vals, ddof=1)),
        "values": psnr_vals.tolist(),
        "mean_ratio": float(np.mean(ratio_vals))
    }

delta_BA = res["B"]["mean"] - res["A"]["mean"]
delta_BC = res["B"]["mean"] - res["C"]["mean"]

print("=" * 85)
print("TỔNG KẾT KHOA HỌC: BẢNG SO SÁNH ĐỐI CHỨNG PRIMARY ENDPOINT (HELD-OUT HYBRID-OOD)")
print("=" * 85)
print(f"Nhánh A (PromptIR Baseline):        {res['A']['mean']:.4f} ± {res['A']['std']:.4f} dB")
print(f"Nhánh B (PHASE-HOPFIELD-IR):       {res['B']['mean']:.4f} ± {res['B']['std']:.4f} dB  (Ratio: {res['B']['mean_ratio']*100:.2f}%)")
print(f"Nhánh C (Param-Matched Control):   {res['C']['mean']:.4f} ± {res['C']['std']:.4f} dB  (Ratio: {res['C']['mean_ratio']*100:.2f}%)")
print("-" * 85)
print(f"-> Chênh lệch Delta(B - A): {delta_BA:+.4f} dB | Yêu cầu Gate X: >= {X:.4f} dB -> {'[PASS CHIẾN THẮNG]' if delta_BA >= X else '[CHƯA ĐẠT]'}")
print(f"-> Chênh lệch Delta(B - C): {delta_BC:+.4f} dB | Yêu cầu Gate Z: >= {Z:.4f} dB -> {'[PASS CHIẾN THẮNG]' if delta_BC >= Z else '[CHƯA ĐẠT]'}")
print(f"-> Hoạt động khối mới:      {res['B']['mean_ratio']*100:.2f}%     | Yêu cầu Gate Ratio: >= 15.00%  -> {'[PASS KHÔNG NGỦ ĐÔNG]' if res['B']['mean_ratio'] >= 0.15 else '[DORMANT]'}")
print("=" * 85)

# Tự động nén và tải về toàn bộ artifacts để nộp báo cáo cho cô Châu
!zip -r phase_hopfield_pilot_results.zip results/ runs/phase_hopfield_v3/ thresholds_locked_v3.json
files.download("phase_hopfield_pilot_results.zip")
